# Data analysis for ingest pipeline

Validate that TasteMender's imported audio-feature data is complete, internally consistent, and faithful to the original AcousticBrainz source data.

This work should identify any issues introduced during ingestion, transformation, normalization, or storage, and establish whether each feature provides enough useful variation to support recommendation ranking.

In [ ]:
import os
import sys
import django
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Enable access to Django project inside current notebook
PROJECT_PATH = os.path.dirname(os.path.abspath(""))
sys.path.append(PROJECT_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "music_recommendation.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

Display genre distribution charts for 2 classifiers: Dortmund and Rosamerica

In [ ]:
from recommend_api.models import GenreDortmund, GenreRosamerica

feature_store = np.load("../features_and_index.npz", allow_pickle=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Pull genre codes for all tracks into a pandas Series
dort = pd.Series(feature_store["genre_dortmund"], name="count")
rosa = pd.Series(feature_store["genre_rosamerica"])

# Count and plot Dortmund, genre labels are stored in Django models
counts_dort = dort.value_counts()
labels_dort = [
    GenreDortmund.objects.filter(code=x).first().label for x in counts_dort.index
]

axes[0].pie(counts_dort, labels=labels_dort, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Genre distribution (Dortmund)")
axes[0].legend(labels_dort, loc="best", fontsize="small")

# Count and plot Rosamerica, genre labels are stored in Django models
counts_rosa = rosa.value_counts()
labels_rosa = [
    GenreRosamerica.objects.filter(code=x).first().label for x in counts_rosa.index
]
axes[1].pie(counts_rosa, labels=labels_rosa, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Genre distribution (Rosamerica)")
axes[1].legend(labels_rosa, loc="best", fontsize="small")

plt.tight_layout()
plt.show()

Count tracks with missing year information

In [ ]:
years = np.array(feature_store["years"])
num_tracks = len(years)
zero_years = np.sum(years == 0)

print(f"Tracks missing a release year: {zero_years:,} / {num_tracks:,}")